# NASA Milling Independent-Dataset Validation — Kaggle

**Purpose:** Evaluate a case-held-out Random Forest baseline and empirical conformal intervals **on NASA Milling data**, not reusing PHM2010 model weights. The NASA `case` variable does **not** prove that each case is a separate physical cutter. This is a comparison of *evaluation protocols* on a different dataset, not a cross-dataset trained-model transfer.

1. Create a Kaggle notebook (CPU is sufficient).
2. Attach the NASA Milling Parquet dataset with a file named `data.parquet`.
3. Run **Cell 1** to save the script, then **Cell 2** to run against the dataset.
4. Download `/kaggle/working/NASA_Independent_Validation.zip`.

The procedure reports empirical case-held-out results; nominal 90% coverage is **not guaranteed** under case-level distribution shift. The script has only been self-tested with synthetic data, not confirmed on your exact downloaded Parquet version.

In [1]:
"""Independent DATASET evaluation protocol: NASA/UC Berkeley Milling, case-held-out.

NOT directly applying PHM-trained models to NASA: incompatible sensors and processes.
This is a replication of the *evaluation procedure* with independently trained baselines.

Input: a Parquet copy of the NASA milling dataset (167 runs, 6 array channels,
case/run/VB etc.), e.g. jonasmaltebecker/nasa_milling on Hugging Face.
NASA original source: https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/
No missing wear labels are interpolated. Units VB are converted from mm to um.
Predeclared: 16 leave-one-case-out folds, with 3 source cases for calibration;
12 source cases for model fitting; no hyperparameter tuning using held-out results.
Conformal empirical test coverage is NOT guaranteed under case-level distribution shift.

Run on Kaggle after attaching data.parquet:
   python nasa_independent_validation.py --data /kaggle/input/<dataset>/data.parquet
Validate code without claiming dataset results:
   python nasa_independent_validation.py --self-test
"""
from __future__ import annotations
import argparse
import hashlib
import json
import zipfile
from pathlib import Path
from typing import Sequence
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SENSORS=('smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle')
META=('run','DOC','feed','material')
RANDOM_SEED=42
ALPHA=.10

def sensor_features(v):
    x=np.asarray(v,dtype=float).reshape(-1)
    x=x[np.isfinite(x)]
    if len(x)<100: raise ValueError('Sensor trace missing or <100 samples')
    med=np.median(x)
    return dict(mean=float(x.mean()),std=float(x.std()),rms=float(np.sqrt(np.mean(x*x))),
        min=float(x.min()),max=float(x.max()),
        p05=float(np.quantile(x,.05)),p95=float(np.quantile(x,.95)),
        median=float(med),mad=float(np.median(np.abs(x-med))))

def prepare(df):
    needed={'case','run','VB',*SENSORS}
    missing=sorted(needed-set(df.columns))
    if missing: raise ValueError(f'Missing NASA Parquet columns: {missing}')
    data=df.copy()
    data['case']=pd.to_numeric(data['case'],errors='raise').astype(int)
    data['run']=pd.to_numeric(data['run'],errors='raise').astype(int)
    data['VB']=pd.to_numeric(data['VB'],errors='coerce')
    if data.duplicated(['case','run']).any():raise AssertionError('Duplicate case/run entries')
    full_n=len(data)
    data=data[data['VB'].notna()].copy()
    data['wear_um']=data['VB']*1000.0
    if not np.isfinite(data['wear_um']).all():raise ValueError('Non-finite wear labels')
    rows=[]
    for _,r in data.iterrows():
        row={'case':int(r['case']),'run':int(r['run']),'wear_um':float(r['wear_um'])}
        for k in ('DOC','feed','material'):
            row[k]=float(r[k]) if k in data.columns and pd.notna(r[k]) else 0.0
        for s in SENSORS:
            f=sensor_features(r[s])
            for name,num in f.items():row[f'{s}_{name}']=num
        rows.append(row)
    feats=pd.DataFrame(rows).sort_values(['case','run']).reset_index(drop=True)
    if feats.isnull().any().any():raise AssertionError('NaN engineered features')
    return feats,full_n

def cp_threshold(residuals, alpha=ALPHA):
    a=np.sort(np.asarray(residuals,dtype=float))
    k=int(np.ceil((len(a)+1)*(1-alpha)))
    return float(a[k-1]) if k<=len(a) else float('inf')

def calculate(feats, verbose=True):
    cases=sorted(feats.case.unique().tolist())
    if len(cases)<6:raise ValueError('Need >=6 independent experimental cases')
    sensor_cols=[c for c in feats.columns if any(c.startswith(s+'_') for s in SENSORS)]
    sources={'Sensor RF':sensor_cols,'Run/condition baseline':list(META)}
    report=[]; pred_rows=[]; fold_defs=[]
    for test in cases:
        others=[x for x in cases if x!=test]
        # Deterministic choice independent of labels; 3 calibration cases.
        pivot=cases.index(test)%len(others)
        circular=others[pivot:]+others[:pivot]
        calcases=circular[:3]
        traincases=[x for x in others if x not in calcases]
        assert set(traincases).isdisjoint(calcases)
        assert test not in traincases and test not in calcases
        tr=feats[feats.case.isin(traincases)].copy()
        cal=feats[feats.case.isin(calcases)].copy()
        te=feats[feats.case==test].copy()
        if min(len(tr),len(cal),len(te))==0:raise ValueError(f'Empty split for case {test}')
        fold_defs.append(dict(Test_case=test,Train_cases=','.join(map(str,traincases)),Calibration_cases=','.join(map(str,calcases)),Train_runs=len(tr),Calibration_runs=len(cal),Test_runs=len(te)))
        for label,col in sources.items():
            rf=RandomForestRegressor(n_estimators=300,max_features='sqrt',min_samples_leaf=3,random_state=RANDOM_SEED,n_jobs=-1)
            rf.fit(tr[col],tr.wear_um)
            yp=rf.predict(te[col]); cal_pred=rf.predict(cal[col]);q=cp_threshold(np.abs(cal.wear_um-cal_pred))
            inside=(np.abs(te.wear_um.to_numpy()-yp)<=q)
            report.append(dict(Test_case=test,Method=label,N_train=len(tr),N_cal=len(cal),N_test=len(te),MAE_um=mean_absolute_error(te.wear_um,yp),RMSE_um=float(np.sqrt(mean_squared_error(te.wear_um,yp))),R2=float(r2_score(te.wear_um,yp)) if len(te)>=2 else float('nan'),q90_um=q,Coverage90_pct=float(inside.mean()*100),MeanWidth_um=2*q))
            for cut,act,p in zip(te.run,te.wear_um,yp): pred_rows.append(dict(Test_case=int(test),Method=label,Run=int(cut),Actual_um=float(act),Predicted_um=float(p),Interval_q_um=q,Lower_um=float(p-q),Upper_um=float(p+q)))
        if verbose:print('Finished case',test,'test rows',len(te),'cal rows',len(cal),flush=True)
    return pd.DataFrame(report),pd.DataFrame(pred_rows),pd.DataFrame(fold_defs)


# NASA EXTERNAL DATASET VALIDATION — EXACT KAGGLE PATH
DATA_PATH=Path('/kaggle/input/datasets/kaziakaid/data-parquet/data.parquet')
OUT=Path('/kaggle/working/NASA_Independent_Validation')
ZIP=Path('/kaggle/working/NASA_Independent_Validation.zip')
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'NASA dataset not found: {DATA_PATH}')
try:
    raw=pd.read_parquet(DATA_PATH)
except ImportError as exc:
    raise ImportError('Install pyarrow in Kaggle, then rerun.') from exc
print('Loaded NASA dataset:',DATA_PATH)
print('Rows:',len(raw))
print('Columns:',list(raw.columns))
print('Extracting features from measured wear runs ...')
feats,total_runs=prepare(raw)
results,predictions,splits=calculate(feats)
OUT.mkdir(parents=True,exist_ok=True)
for name,value in [('case_results.csv',results),('case_predictions.csv',predictions),('case_splits.csv',splits),('engineered_measured_runs.csv',feats)]:
    value.to_csv(OUT/name,index=False)
macro=results.groupby('Method',as_index=False).agg(
    Macro_case_MAE_um=('MAE_um','mean'),
    Macro_case_coverage_pct=('Coverage90_pct','mean'),
    Macro_case_width_um=('MeanWidth_um','mean'),
    Test_cases=('Test_case','nunique'))
macro.to_csv(OUT/'macro_summary.csv',index=False)
manifest={'data_path':str(DATA_PATH),'total_source_runs':int(total_runs),
    'measured_labeled_runs':int(len(feats)),'number_of_cases':int(feats.case.nunique()),
    'wear_label':'VB originally in mm, converted to micrometres',
    'evaluation':'leave-one-NASA-case-out; three source cases for calibration',
    'model':'RandomForestRegressor(300, max_features=sqrt, min_samples_leaf=3, random_state=42)',
    'conformal_nominal_coverage':.90,
    'limitations':['No PHM2010 model weights transferred to NASA',
        'NASA cases do not necessarily identify separate physical cutters',
        'No conformal guarantee under case-level distribution shift',
        'Missing VB labels excluded without interpolation']}
(OUT/'protocol_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for p in sorted(OUT.glob('*')):
        if p.is_file(): z.write(p,arcname=f'{OUT.name}/{p.name}')
with zipfile.ZipFile(ZIP,'r') as z:
    assert z.testzip() is None,'ZIP integrity check failed'
print('\n=== NASA EXTERNAL VALIDATION FINISHED ===')
print(macro.round(3).to_string(index=False))
print('\nResults directory:',OUT)
print('Download ZIP:',ZIP)
print('ZIP size (MB):',round(ZIP.stat().st_size/1024**2,2))


Loaded NASA dataset: /kaggle/input/datasets/kaziakaid/data-parquet/data.parquet
Rows: 167
Columns: ['case', 'run', 'VB', 'time', 'DOC', 'feed', 'material', 'smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']
Extracting features from measured wear runs ...
Finished case 1 test rows 13 cal rows 34
Finished case 2 test rows 13 cal rows 27
Finished case 3 test rows 14 cal rows 14
Finished case 4 test rows 7 cal rows 14
Finished case 5 test rows 6 cal rows 13
Finished case 6 test rows 1 cal rows 21
Finished case 7 test rows 7 cal rows 24
Finished case 8 test rows 5 cal rows 39
Finished case 9 test rows 9 cal rows 42
Finished case 10 test rows 10 cal rows 45
Finished case 11 test rows 20 cal rows 32
Finished case 12 test rows 12 cal rows 26
Finished case 13 test rows 13 cal rows 16
Finished case 14 test rows 7 cal rows 22
Finished case 15 test rows 6 cal rows 29
Finished case 16 test rows 3 cal rows 40

=== NASA EXTERNAL VALIDATION FINISHED ===
                Method  Mac

If the script raises **missing-column**, **invalid-array**, or **package** errors, share the complete error output. Do not edit target labels, split logic, or the evaluation protocol after inspecting held-out outcomes without recording a new exploratory version.